# 卍 The Spherical Manifold: 12-Channel Deterministic Ecosystem (Kaggle Prototype)
### System Anchoring: MANIFOLD / Vinculum Compiler Pipeline

This notebook represents the **Phase 1 Prototyping Substrate** for simulating, validating, and optimizing a closed-topology spherical planet surface. Instead of flat Cartesian planes that hit coordinate borders, we parameterized a 6-face Cubemap grid ($6 \times N \times N$) with a 12-channel tensor structure.

## 📐 Tensor Spec v2.0 (12 Channels, 48-byte aligned)
- **Channels 0-5 (Conservative):** `rock`, `soil`, `sand`, `water`, `ice`, `organic`
- **Channels 6-10 (Additive):** `biomass_prey`, `biomass_pred`, `spore_density`, `terrain_stress`, `thermal_flux`
- **Channel 11 (WebGPU Padding):** `_pad` (WebGPU struct memory boundaries)

In [ ]:
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt

N = 16
CHANNELS = 12

ROCK = 0
SOIL = 1
SAND = 2
WATER = 3
ICE = 4
ORGANIC = 5
BIOMASS_PREY = 6
BIOMASS_PRED = 7
SPORE_DENSITY = 8
TERRAIN_STRESS = 9
THERMAL_FLUX = 10
PAD = 11

## 🧬 Vinculum Static Compiler (Python Port)
Resolves the topological execution sequence and flags write-write race conditions.

In [ ]:
class VinculumCompiler:
    def __init__(self, spec):
        self.spec = spec
        self.dag = {}
        self.build_dag()
        self.validate_no_races()

    def build_dag(self):
        for name, mod in self.spec["modules"].items():
            self.dag[name] = set(mod.get("requires", []))
            if mod.get("flux_producer") and ("water" in mod["writes"] or "sand" in mod["writes"]):
                cons_mod = self.spec["modules"].get("semi_implicit_conservation")
                if cons_mod and name not in cons_mod.get("requires", []):
                    cons_mod.setdefault("requires", []).append(name)
                    self.dag["semi_implicit_conservation"] = set(cons_mod["requires"])

    def validate_no_races(self):
        write_map = {}
        for name, mod in self.spec["modules"].items():
            for ch in mod["writes"]:
                write_map.setdefault(ch, []).append(name)
        for ch, writers in write_map.items():
            if len(writers) > 1:
                for i in range(len(writers)):
                    for j in range(i+1, len(writers)):
                        if not self.is_ordered(writers[i], writers[j]) and not self.is_ordered(writers[j], writers[i]):
                            raise ValueError(f"Race condition on channel '{ch}': {writers[i]} and {writers[j]} without DAG edge!")

    def is_ordered(self, a, b):
        visited, stack = set(), [a]
        while stack:
            curr = stack.pop()
            if curr == b: return True
            if curr in visited: continue
            visited.add(curr)
            stack.extend(self.dag.get(curr, []))
        return False

    def topo_sort(self):
        in_degree = {mod: 0 for mod in self.spec["modules"]}
        for mod, deps in self.dag.items():
            for dep in deps:
                if dep in in_degree: in_degree[mod] += 1
        queue = [mod for mod, deg in in_degree.items() if deg == 0]
        result = []
        while queue:
            curr = queue.pop(0)
            result.append(curr)
            for mod, deps in self.dag.items():
                if curr in deps:
                    in_degree[mod] -= 1
                    if in_degree[mod] == 0: queue.append(mod)
        if len(result) != len(self.spec["modules"]): raise ValueError("Cycle detected in dependency graph!")
        return result

## 🌐 Orthonormal Geodesic Joint Mapping (Cubemap Transitions)
We define direction vectors for all cells, and map offsets dynamically to create **tear-free, polar-singularity-free** neighborhood lookups on the sphere using 3D projections.

In [ ]:
class SphericalTensorSim:
    def __init__(self, N=16):
        self.N = N
        self.state = np.zeros((6, N, N, CHANNELS), dtype=np.float32)
        self.precompute_directions_and_neighbors()
        self.reset_to_base_terrain()

    def face_uv_to_dir(self, face, u, v):
        u_val = u * 2.0 - 1.0
        v_val = v * 2.0 - 1.0
        if face == 0:   return np.array([1.0, -v_val, -u_val])
        elif face == 1: return np.array([-1.0, -v_val, u_val])
        elif face == 2: return np.array([u_val, 1.0, v_val])
        elif face == 3: return np.array([u_val, -1.0, -v_val])
        elif face == 4: return np.array([u_val, -v_val, 1.0])
        elif face == 5: return np.array([-u_val, -v_val, -1.0])
        return np.array([0.0, 0.0, 0.0])

    def precompute_directions_and_neighbors(self):
        self.dirs = np.zeros((6, self.N, self.N, 3), dtype=np.float32)
        for f in range(6):
            for r in range(self.N):
                for c in range(self.N):
                    self.dirs[f, r, c] = self.face_uv_to_dir(f, (c+0.5)/self.N, (r+0.5)/self.N)
                    self.dirs[f, r, c] /= np.linalg.norm(self.dirs[f, r, c])
        
        flat_dirs = self.dirs.reshape(-1, 3)
        self.neighbor_indices = np.zeros((6, self.N, self.N, 4, 3), dtype=np.int32)
        
        epsilon = 1.5 / self.N
        for f in range(6):
            for r in range(self.N):
                for c in range(self.N):
                    V = self.dirs[f, r, c]
                    T1 = np.cross(V, [0, 0, 1]) if abs(V[2]) < 0.9 else np.cross(V, [1, 0, 0])
                    T1 /= np.linalg.norm(T1)
                    T2 = np.cross(V, T1)
                    T2 /= np.linalg.norm(T2)

                    neighbors_dirs = [V + epsilon*T2, V - epsilon*T2, V + epsilon*T1, V - epsilon*T1]
                    for d_idx, ndir in enumerate(neighbors_dirs):
                        ndir /= np.linalg.norm(ndir)
                        best_idx = np.argmax(np.dot(flat_dirs, ndir))
                        nf = best_idx // (self.N**2)
                        rem = best_idx % (self.N**2)
                        self.neighbor_indices[f, r, c, d_idx] = [nf, rem // self.N, rem % self.N]

    def reset_to_base_terrain(self):
        for f in range(6):
            for r in range(self.N):
                for c in range(self.N):
                    V = self.dirs[f, r, c]
                    rock_height = 30.0 + 4.0*math.sin(V[0]*3)*math.cos(V[1]*3) + 2.0*math.sin(V[2]*6)
                    self.state[f, r, c, ROCK] = max(1.0, rock_height)
                    self.state[f, r, c, SOIL] = 3.0 + 2.0*max(0.0, V[2])
                    self.state[f, r, c, SAND] = 1.0 + 1.0*max(0.0, -V[1])
                    self.state[f, r, c, WATER] = max(0.1, 29.0 - rock_height) if rock_height < 29.0 else 0.1
                    self.state[f, r, c, ORGANIC] = 2.0 + max(0.0, V[0])
                    self.state[f, r, c, BIOMASS_PREY] = 0.5 if rock_height >= 29.0 else 0.0
                    self.state[f, r, c, BIOMASS_PRED] = 0.1 if rock_height >= 29.0 else 0.0
                    self.state[f, r, c, SPORE_DENSITY] = 0.2
                    self.state[f, r, c, THERMAL_FLUX] = 20.0 + 10.0*V[1]

## 💥 CSG Terraforming, Geodesic Advection, and Lotka-Volterra Physics
Implement core simulation equations.

In [ ]:
def simulate_csg_crater(sim, lat, lon, radius=5.0, depth=10.0):
    lat_r, lon_r = math.radians(lat), math.radians(lon)
    target = np.array([math.cos(lat_r)*math.cos(lon_r), math.sin(lat_r), math.cos(lat_r)*math.sin(lon_r)])
    for f in range(6):
        for r in range(sim.N):
            for c in range(sim.N):
                V = sim.dirs[f, r, c]
                dist = np.linalg.norm(V * 40.0 - target * 40.0)
                if dist < radius:
                    factor = 1.0 - (dist / radius)**2
                    excavation = depth * factor
                    # Excavate sediment & soil first
                    for ch in [SAND, SOIL, ROCK]:
                        val = sim.state[f, r, c, ch]
                        if val >= excavation:
                            sim.state[f, r, c, ch] -= excavation
                            break
                        else:
                            excavation -= val
                            sim.state[f, r, c, ch] = 0.0
                    sim.state[f, r, c, TERRAIN_STRESS] += 2.5 * factor
                    sim.state[f, r, c, THERMAL_FLUX] += 50.0 * factor

def simulate_advection(sim, dt=0.1):
    new_water = np.copy(sim.state[..., WATER])
    for f in range(6):
        for r in range(sim.N):
            for c in range(sim.N):
                w = sim.state[f, r, c, WATER]
                if w < 0.01: continue
                tot_h = np.sum(sim.state[f, r, c, ROCK:WATER+1])
                flows = []
                for d in range(4):
                    nf, nr, nc = sim.neighbor_indices[f, r, c, d]
                    n_tot_h = np.sum(sim.state[nf, nr, nc, ROCK:WATER+1])
                    diff = tot_h - n_tot_h
                    if diff > 0: flows.append((diff, nf, nr, nc))
                if not flows: continue
                tot_diff = sum(fd[0] for fd in flows)
                for diff, nf, nr, nc in flows:
                    amount = min(w * 0.5 * (diff / tot_diff), w * dt)
                    new_water[f, r, c] -= amount
                    new_water[nf, nr, nc] += amount
                    # Sand advection
                    sand_wash = amount * 0.1 * sim.state[f, r, c, SAND]
                    sim.state[f, r, c, SAND] -= sand_wash
                    sim.state[nf, nr, nc, SAND] += sand_wash
    sim.state[..., WATER] = new_water

def simulate_lotka_volterra(sim, dt=0.1):
    alpha, beta, gamma = 0.6, 0.45, 0.35
    for f in range(6):
        for r in range(sim.N):
            for c in range(sim.N):
                prey = sim.state[f, r, c, BIOMASS_PREY]
                pred = sim.state[f, r, c, BIOMASS_PRED]
                if prey <= 0.001 and pred <= 0.001: continue
                carrying_cap = 2.0 * max(0.05, sim.state[f, r, c, WATER] * sim.state[f, r, c, ORGANIC])
                d_prey = alpha * prey * (1.0 - prey/carrying_cap) - beta * prey * pred
                d_pred = beta * prey * pred - gamma * pred
                sim.state[f, r, c, BIOMASS_PREY] = max(0.0, prey + dt*d_prey)
                sim.state[f, r, c, BIOMASS_PRED] = max(0.0, pred + dt*d_pred)
                decay = (prey * 0.05 + pred * 0.08) * dt
                sim.state[f, r, c, ORGANIC] = min(10.0, sim.state[f, r, c, ORGANIC] + decay)

## 🖥️ Execute Compiled Pipeline & Plot 6-Face Cubemap Topology

In [ ]:
sim = SphericalTensorSim(N=N)
initial_mass = np.sum(sim.state[..., ROCK:ORGANIC+1])
print(f"Initial Global Conservative Mass: {initial_mass:.4f} units")

# Excavate two craters
simulate_csg_crater(sim, lat=30, lon=45, radius=5.0, depth=8.0)
simulate_csg_crater(sim, lat=-20, lon=120, radius=4.0, depth=6.0)

# Run advection and Lotka-Volterra
for _ in range(30):
    simulate_advection(sim)
    simulate_lotka_volterra(sim)

current_mass = np.sum(sim.state[..., ROCK:ORGANIC+1])
print(f"Final Global Conservative Mass: {current_mass:.4f} units")
print(f"Absolute Mass Deviation (Symbolic Leak): {abs(current_mass - initial_mass):.6f}")

## 🗺️ Plotting the 6 Cubemap Faces (Rock Elevation Map)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
faces_names = ["+X (Face 0)", "-X (Face 1)", "+Y (Face 2)", "-Y (Face 3)", "+Z (Face 4)", "-Z (Face 5)"]
for f in range(6):
    ax = axes[f // 3, f % 3]
    im = ax.imshow(sim.state[f, :, :, ROCK], cmap="terrain", origin="lower")
    ax.set_title(faces_names[f])
    fig.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

## 🏛️ Export Seed
We now write out `game/spherical_seed.json` for loading direct displacements into the Three.js mesh.

In [ ]:
seed_data = []
for f in range(6):
    face_list = []
    for r in range(N):
        row_list = []
        for c in range(N):
            row_list.append({
                "rock": float(sim.state[f, r, c, ROCK]),
                "soil": float(sim.state[f, r, c, SOIL]),
                "sand": float(sim.state[f, r, c, SAND]),
                "water": float(sim.state[f, r, c, WATER]),
                "ice": float(sim.state[f, r, c, ICE]),
                "organic": float(sim.state[f, r, c, ORGANIC]),
                "prey": float(sim.state[f, r, c, BIOMASS_PREY]),
                "pred": float(sim.state[f, r, c, BIOMASS_PRED]),
                "spore": float(sim.state[f, r, c, SPORE_DENSITY]),
                "stress": float(sim.state[f, r, c, TERRAIN_STRESS]),
                "flux": float(sim.state[f, r, c, THERMAL_FLUX]),
                "dir": [float(x) for x in sim.dirs[f, r, c]]
            })
        face_list.append(row_list)
    seed_data.append(face_list)

seed_path = "../game/spherical_seed.json"
with open(seed_path, "w") as f:
    json.dump({"N": N, "faces": seed_data}, f, indent=2)
print(f"Seeding complete. Seed saved to: {os.path.abspath(seed_path)}")